# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. You'll learn how to examine metadata, explore record sets and their fields (all by `@id`), extract tabular data, perform exploratory data analysis, and visualize core variables.

### Dataset Source
The dataset schema is provided as a Croissant JSON-LD file at the following URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```
This dataset describes 77 cancer survivors with second primary colorectal cancer, including diverse clinical and molecular fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and records
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets (`RecordSet`) and their fields (`Field`) by `@id`.

The `mlcroissant` interface allows inspection of all record sets defined in the schema using the `.record_sets` attribute. For each record set, its `@id`, human-readable name and included field `@id`s are displayed.

In [ ]:
# List all available record sets and their fields (by `@id`)
print('Available record sets:')
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    name = record_set.get('name', '(no name)')
    print(f"  Name: {name}")
    fields = record_set.get('field', [])
    # Ensure field is a list
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id', field)}")
        else:
            print(f"    - {field}")
    record_set_ids.append(record_set['@id'])
if not record_set_ids:
    print("No record sets found. Double-check the Croissant schema.")

## 3. Data Extraction

Let's load data from each available record set. All references are by `@id`, following the Croissant schema specification. If there are multiple record sets, they will be loaded into separate dataframes.

You can explore the dataframe columns to see which fields each record set contains.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames
dataframes = {}
# If no record sets detected, indicate and stop
if len(record_set_ids) == 0:
    print("No record sets to extract data from.")
else:
    for rs_id in record_set_ids:
        print(f"Loading records from RecordSet @id: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records; columns: {list(df.columns)}")
        except Exception as e:
            print(f"  Failed to load records from {rs_id}: {e}")
    # For exploration, print columns and preview for the first record set found
    if record_set_ids:
        sample_rs = record_set_ids[0]
        print(f"\nFields/columns for RecordSet '{sample_rs}':\n", dataframes[sample_rs].columns.tolist())
        display(dataframes[sample_rs].head())

## 4. Exploratory Data Analysis (EDA)

In this section, we perform basic EDA steps on the main tabular record set. We'll:
- Select a numeric field (by its `@id`) for analysis,
- Filter records based on a threshold,
- Normalize that numeric field,
- Optionally group by a key attribute if appropriate (for categorical fields).

You can adapt the field `@id`s here based on your dataset exploration above.

In [ ]:
# --- EDA on a main record set ---

# Choose the primary record set for EDA (change as needed)
main_rs_id = record_set_ids[0] if record_set_ids else None

if main_rs_id is None or main_rs_id not in dataframes:
    print("No main record set found for EDA.")
else:
    df = dataframes[main_rs_id].copy()
    print(f"Working with record set: {main_rs_id}")
    print(f"Available fields: {df.columns.tolist()}")
    
    # Try to find a numeric field to analyze, fallback to 'age' or similar if present
    import numpy as np
    numeric_field = None
    for col in df.columns:
        # sanitize data to numeric for detection
        vals = pd.to_numeric(df[col], errors='coerce')
        if vals.notna().sum() > 0 and (vals.dtype == np.float64 or vals.dtype == np.int64):
            if len(np.unique(vals.dropna())) > 5:
                numeric_field = col
                break
    if numeric_field is None:
        print("No suitable numeric field detected. Unable to proceed with EDA example.")
    else:
        print(f"Using numeric field for analysis: '{numeric_field}' (likely a variable like age)")
        
        # Convert to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.3) if df[numeric_field].notna().sum() > 10 else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try to group by a likely categorical field, e.g., 'sex', 'gender', or the first non-numeric field
        cat_field = None
        for col in df.columns:
            if col.lower() in ['sex', 'gender', 'anatomical_location', 'msi_status'] and df[col].nunique() <= 10:
                cat_field = col
                break
        if cat_field is None:
            # fallback: pick first non-numeric, low-cardinality field
            for col in df.columns:
                if df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() <= 10:
                    cat_field = col
                    break
        if cat_field:
            print(f"Grouping by '{cat_field}':")
            grouped_df = filtered_df.groupby(cat_field)[numeric_field].mean().to_frame('mean').join(
                filtered_df.groupby(cat_field)[numeric_field].count().to_frame('n')
            )
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with a categorical variable, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is None or main_rs_id not in dataframes or numeric_field is None:
    print("Visualization cannot be created: No numeric field or data for main record set.")
else:
    # Plot distribution of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If categorical field exists, plot boxplot
    if cat_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=cat_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {cat_field}")
        plt.show()

## 6. Conclusion

In this notebook, you used `mlcroissant` to inspect, extract, and explore a FAIR² clinical dataset defined by a Croissant schema. By referencing all entities by their `@id`, you:
- Loaded structured metadata and tabular records
- Explored available record sets and fields
- Extracted main study variables for analysis
- Carried out basic filtering, normalization, and grouping
- Visualized numeric and categorical variables interactively

This approach can be adapted for any dataset compliant with the Croissant standard and the `mlcroissant` library. For more advanced analysis, you can extend the EDA/visualization sections or combine multiple record sets. Refer to the dataset's Croissant JSON-LD for full schema details and standardized entity `@id`s.